# Quito, a Weekday and a Weekend Day

Each day as one O-D matrix, assigned once against capacity scaled up to a day.

- **Weekday:** the road model's morning peak, turned into a whole day.
- **Weekend day:** the average of a Saturday and a Sunday. A gravity model sends its trips to
  **weekend attractions**: shops and the historic centre pull harder, workplaces and schools less.

Reads what `TAZ_OD_demo.ipynb` exported and rebuilds none of it:

| needs | from |
|---|---|
| 761 zones, with floor area by class | `outputs/taz_od_exports/zones.gpkg` |
| O-D matrix | `outputs/taz_od_exports/od_matrix.omx` |
| its assumptions | `outputs/taz_od_exports/assumptions.csv` |
| network and connectors | `data/road_model/` |

The exports are committed. `data/road_model/` is not: it is written by a run of
`TAZ_OD_demo.ipynb`, which needs `data/gov_data/`.

In [ ]:
import ast
import os
import shutil
import tempfile

import folium
import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

_cwd = os.getcwd()
BASE_DIR = _cwd if os.path.isdir(os.path.join(_cwd, "outputs")) else os.path.dirname(_cwd)
EXPORTS = os.path.join(BASE_DIR, "outputs", "taz_od_exports")
MAPS_DIR = os.path.join(BASE_DIR, "outputs", "weekday_weekend_maps")
EXPORT_DIR = os.path.join(BASE_DIR, "outputs", "weekday_weekend_exports")
for _folder in (MAPS_DIR, EXPORT_DIR):
    os.makedirs(_folder, exist_ok=True)
    for _stale in os.listdir(_folder):
        _path = os.path.join(_folder, _stale)
        if os.path.isfile(_path):
            os.remove(_path)

SOURCE_MODEL = os.path.join(BASE_DIR, "data", "road_model")
WORK_MODEL = os.environ.get("WEEK_MODEL_DIR",
                            os.path.join(BASE_DIR, "data", "weekday_weekend_model"))
METRE_CRS = "EPSG:32717"

for _needed in (os.path.join(EXPORTS, "zones.gpkg"),
                os.path.join(EXPORTS, "od_matrix.omx"),
                os.path.join(EXPORTS, "assumptions.csv"),
                os.path.join(SOURCE_MODEL, "project_database.sqlite")):
    if not os.path.exists(_needed):
        raise FileNotFoundError(f"Missing input: {_needed}\n"
                                f"Run TAZ_OD_demo.ipynb first - it writes these.")

print(f"exports: {EXPORTS}\nmodel:   {WORK_MODEL}\n"
      f"maps:    {MAPS_DIR}\nmatrices: {EXPORT_DIR}")

## A1. Inputs

The zones, the road model's morning peak hour `T` (from home to activity), and three of the road
model's assumptions: the intrazonal cost, the average trip length and the attraction rates.

In [ ]:
from aequilibrae.matrix import AequilibraeMatrix                       # noqa: E402

zones = gpd.read_file(os.path.join(EXPORTS, "zones.gpkg")).sort_values("zone_number")
zones = zones.reset_index(drop=True)

_read = AequilibraeMatrix()
_read.create_from_omx(os.path.join(EXPORTS, "od_matrix.omx"),
                      os.path.join(WORK_MODEL + "_tmp.aem"), memory_only=True)
T = np.array(_read.matrix["car"], dtype=float)
ZONE_IDS = np.array(_read.index[:], dtype=np.int64)
_read.close()
if os.path.exists(WORK_MODEL + "_tmp.aem"):
    os.remove(WORK_MODEL + "_tmp.aem")
if not np.array_equal(ZONE_IDS, zones["zone_number"].to_numpy()):
    raise ValueError("the matrix and the zone table are not in the same order")

_assumed = pd.read_csv(os.path.join(EXPORTS, "assumptions.csv"), index_col=0)


def taz_assumption(name):
    return ast.literal_eval(str(_assumed.loc[name, "value"]))


INTRAZONAL_SHARE = taz_assumption("INTRAZONAL_SHARE")
MEAN_TRIP_MIN = taz_assumption("MEAN_TRIP_MIN")
WEEKDAY_RATES = taz_assumption("ATTRACTION_RATES")

print(f"{len(zones):,} zones; morning peak hour: {T.sum():,.0f} vehicle trips")
print(f"intrazonal share {INTRAZONAL_SHARE}, average trip {MEAN_TRIP_MIN} min")
print(f"weekday attraction rates: {WEEKDAY_RATES}")

## A2. The road model

A copy, so the runs writing results leave `data/road_model/` untouched. Free-flow travel time
between zones, for the gravity model, with the diagonal filled as the road model fills it: a share
of the trip to the nearest other zone.

In [ ]:
import logging                                                        # noqa: E402

from aequilibrae import Project                                       # noqa: E402
from aequilibrae.paths import NetworkSkimming                         # noqa: E402


def _release_log_files(folder):
    """AequilibraE keeps a log file open inside the project; Windows will not delete it."""
    target = os.path.abspath(folder)
    for logger in [logging.getLogger()] + [logging.getLogger(n) for n
                                           in list(logging.Logger.manager.loggerDict)]:
        for handler in list(getattr(logger, "handlers", [])):
            written = getattr(handler, "baseFilename", None)
            if written and os.path.abspath(written).startswith(target):
                handler.close()
                logger.removeHandler(handler)


def open_working_copy():
    """Copy the saved model and open the copy."""
    if os.path.isdir(WORK_MODEL):
        _release_log_files(WORK_MODEL)
        try:
            shutil.rmtree(WORK_MODEL)
        except PermissionError as broken:
            raise PermissionError(
                f"{WORK_MODEL} is held open by another process, so it cannot be replaced. "
                f"Close any notebook using it, or set WEEK_MODEL_DIR to another path."
            ) from broken
    shutil.copytree(SOURCE_MODEL, WORK_MODEL,
                    ignore=shutil.ignore_patterns("*.log", "osm_data"))
    project = Project()
    project.open(WORK_MODEL)
    return project


model = open_working_copy()
model.network.build_graphs(modes=["c"])
car_graph = model.network.graphs["c"]
car_graph.prepare_graph(ZONE_IDS)
car_graph.set_graph("travel_time")
car_graph.set_skimming(["travel_time"])
car_graph.set_blocked_centroid_flows(True)

_skimming = NetworkSkimming(car_graph)
_skimming.execute()
cost = np.array(_skimming.results.skims.matrix["travel_time"], dtype=float).copy()
if not np.array_equal(np.array(_skimming.results.skims.index).astype(int), ZONE_IDS):
    raise ValueError("the skim is not in zone order")

_reachable = np.isfinite(cost) & (cost > 0)
np.fill_diagonal(_reachable, False)
_connected = _reachable.sum(axis=1) > 0
_nearest = np.full(len(cost), np.nan)
_nearest[_connected] = np.nanmin(np.where(_reachable, cost, np.nan)[_connected], axis=1)
np.fill_diagonal(cost, np.nan_to_num(_nearest * INTRAZONAL_SHARE))
IMPEDANCE = np.where(np.isfinite(cost), cost, np.nanmax(cost[np.isfinite(cost)]) * 10)
print(f"graph ready: {car_graph.compact_num_links:,} compact links, {len(ZONE_IDS):,} centroids")
print(f"skim: median {np.median(cost[_reachable]):.1f} min between zones")

## B1. The weekday matrix

Over a whole day every trip out has a trip home, so the day's matrix is symmetric:

`weekday = WEEKDAY_SIZE × (T + T') / 2`

`WEEKDAY_SIZE` is the weekday's traffic in morning-peak hours. It is not measured.

In [ ]:
# A weekday's traffic in morning-peak hours.
WEEKDAY_SIZE = 10.16

DAYS = ["weekday", "weekend day"]
DAY_MATRIX = {"weekday": WEEKDAY_SIZE * (T + T.T) / 2}
print(f"weekday: {DAY_MATRIX['weekday'].sum():,.0f} vehicle trips")

## B2. The gravity model

AequilibraE's `GravityApplication`, as the road model runs it: `EXPO` deterrence, with `beta`
solved on the weekday's productions and attractions so the average trip lasts `MEAN_TRIP_MIN`.

In [ ]:
from aequilibrae.distribution import GravityApplication, SyntheticGravityModel   # noqa: E402

PRODUCTIONS = zones["veh_productions"].to_numpy()


def gravity_matrix(attractions, beta):
    """One run of the gravity model: productions fixed, attractions rescaled to match them."""
    vectors = pd.DataFrame({"productions": PRODUCTIONS,
                            "attractions": attractions * PRODUCTIONS.sum() / attractions.sum()},
                           index=ZONE_IDS)
    impedance = AequilibraeMatrix()
    impedance.create_empty(zones=len(ZONE_IDS), matrix_names=["time"], memory_only=True)
    impedance.index[:] = ZONE_IDS
    impedance.matrices[:, :, 0] = IMPEDANCE
    impedance.computational_view(["time"])

    shape = SyntheticGravityModel()
    shape.function = "EXPO"
    shape.beta = float(beta)
    run = GravityApplication(impedance=impedance, vectors=vectors, row_field="productions",
                             column_field="attractions", model=shape, nan_as_zero=True)
    run.apply()
    return np.array(run.output.matrix["gravity"], dtype=float)


def mean_trip_minutes(trips):
    """Trip-weighted average travel time, over the pairs the network can serve."""
    usable = np.isfinite(cost) & (trips > 0)
    return float((trips[usable] * cost[usable]).sum() / trips[usable].sum())


def solve_beta(attractions, target):
    """Bisection on beta: more deterrence means shorter trips."""
    low, high = 0.001, 1.0
    for _ in range(24):
        mid = (low + high) / 2
        if mean_trip_minutes(gravity_matrix(attractions, mid)) > target:
            low = mid
        else:
            high = mid
    return (low + high) / 2


BETA = solve_beta(zones["veh_attractions"].to_numpy(), MEAN_TRIP_MIN)
print(f"beta = {BETA:.4f}")

## B3. The weekend day's matrix

Same floor area, other rates per m². Productions and `beta` stay as on a weekday: the same people
leave home, with the same tolerance for travel time. Only where they go changes.

`weekend day = DAY_SIZE × (WEEKEND + WEEKEND') / 2`

`DAY_SIZE` is a weekday's `WEEKDAY_SIZE`, times the average of a Saturday and a Sunday. None of
these numbers is measured.

In [ ]:
# Relative pull per m2 of floor area on a weekend. Only the ratios matter.
WEEKEND_RATES = {"mixed/commercial": 1.00, "facilities": 0.40, "heritage core": 1.00,
                 "industrial": 0.05, "residential": 0.15}
CLASS_COLUMNS = {"mixed/commercial": "mixed_commercial", "facilities": "facilities",
                 "heritage core": "heritage_core", "industrial": "industrial",
                 "residential": "residential"}
# A Saturday's and a Sunday's traffic, against a weekday's.
SATURDAY, SUNDAY = 0.80, 0.65


def pull(rates):
    """Each zone's attraction: its floor area by class, weighted by the rates."""
    return sum(zones[col].to_numpy() * rates[cls] for cls, col in CLASS_COLUMNS.items())


_weekday_pull = pull(WEEKDAY_RATES)
if not np.allclose(_weekday_pull / _weekday_pull.sum(),
                   zones["veh_attractions"] / zones["veh_attractions"].sum()):
    raise ValueError("the weekday rates do not reproduce the exported attractions")

WEEKEND = gravity_matrix(pull(WEEKEND_RATES), BETA)
DAY_SIZE = WEEKDAY_SIZE * (SATURDAY + SUNDAY) / 2
DAY_MATRIX["weekend day"] = DAY_SIZE * (WEEKEND + WEEKEND.T) / 2

print(f"  {'class':<18}{'weekday':>9}{'weekend':>9}   share of all attraction")
_wd_total, _we_total = _weekday_pull.sum(), pull(WEEKEND_RATES).sum()
for _cls, _col in CLASS_COLUMNS.items():
    print(f"  {_cls:<18}{WEEKDAY_RATES[_cls]:>9.2f}{WEEKEND_RATES[_cls]:>9.2f}   "
          f"{100 * (zones[_col] * WEEKDAY_RATES[_cls]).sum() / _wd_total:5.1f}% -> "
          f"{100 * (zones[_col] * WEEKEND_RATES[_cls]).sum() / _we_total:5.1f}%")
print(f"\nweekend matrix: average trip {mean_trip_minutes(WEEKEND):.1f} min")
print(f"weekend day: {DAY_MATRIX['weekend day'].sum():,.0f} vehicle trips")

## C1. Daily capacity

A link's capacity is vehicles per hour. Multiplied by `F` it becomes vehicles per day, so a daily
matrix can be assigned against it:

`F = 1 / BUSIEST_HOUR_SHARE`

At that `F` the day is assigned as if its busiest hour were repeated all day long. On a weekday
the busiest hour is the morning peak, so its share is `1 / WEEKDAY_SIZE`. Each day's daily
capacity is one more column on the graph; the network itself is not changed.

In [ ]:
# The share of a day's traffic in its busiest hour.
BUSIEST_HOUR_SHARE = {"weekday": 1 / WEEKDAY_SIZE, "weekend day": 0.077}
CAPACITY_FACTOR = {d: 1 / BUSIEST_HOUR_SHARE[d] for d in DAYS}


def capacity_column(day):
    return "capacity_" + day.replace(" ", "_")


for _d in DAYS:
    car_graph.graph[capacity_column(_d)] = car_graph.graph["capacity"] * CAPACITY_FACTOR[_d]
    print(f"  {_d:<13}busiest hour {100 * BUSIEST_HOUR_SHARE[_d]:.1f}% of the day, "
          f"F = {CAPACITY_FACTOR[_d]:.2f}")

## C2. The assignments

User equilibrium with BPR, one run per day, each against its own capacity column named in
`set_capacity_field()`. The flows that come out are vehicles per day.

In [ ]:
from aequilibrae.paths import TrafficAssignment, TrafficClass         # noqa: E402

BPR_ALPHA, BPR_BETA = 0.15, 4.0

RGAP_TARGET, MAX_ITER = 0.001, 200


def as_matrix(array, name="car"):
    """Wrap a zone-by-zone array as the demand matrix the solver reads."""
    holder = AequilibraeMatrix()
    holder.create_empty(zones=array.shape[0], matrix_names=[name], memory_only=True)
    holder.index[:] = ZONE_IDS
    holder.matrices[:, :, 0] = array
    holder.computational_view([name])
    return holder


def assign_day(day):
    """One user-equilibrium assignment of a day's matrix against its daily capacity."""
    job = TrafficAssignment()
    job.add_class(TrafficClass(name="car", graph=car_graph, matrix=as_matrix(DAY_MATRIX[day])))
    job.set_vdf("BPR")
    job.set_vdf_parameters({"alpha": BPR_ALPHA, "beta": BPR_BETA})
    job.set_capacity_field(capacity_column(day))
    job.set_time_field("travel_time")
    job.set_algorithm("bfw")
    job.max_iter = MAX_ITER
    job.rgap_target = RGAP_TARGET
    job.execute()
    job.save_results(table_name=day.replace(" ", "_"))
    print(f"{day:<13}gap {job.assignment.rgap:.2e} after {job.assignment.iter:>3} iterations"
          + ("" if job.assignment.rgap <= RGAP_TARGET else "   ** did not reach the target"))
    return job.results()


results = {d: assign_day(d) for d in DAYS}
print(f"\nresults saved in {os.path.join(WORK_MODEL, 'results_database.sqlite')}")

## C3. The two days

V/C is daily volume over daily capacity: the same ratio the busiest hour would have.

In [ ]:
CAR_AB, CAR_BA = "car_ab", "car_ba"

with model.db_connection as _conn:
    links = pd.read_sql(
        "SELECT link_id, name, link_type, direction, distance, capacity_ab, capacity_ba, "
        "travel_time_ab, travel_time_ba, lanes_ab, speed_ab FROM links", _conn)


def streets(day):
    """A day's link results joined to the network, connectors left out, daily V/C."""
    out = results[day].rename_axis("link_id").reset_index().merge(links, on="link_id",
                                                                  how="left")
    out = out[out["link_type"] != "centroid_connector"].set_index("link_id")
    for col in (CAR_AB, CAR_BA, "Congested_Time_AB", "Congested_Time_BA",
                "travel_time_ab", "travel_time_ba"):
        out[col] = out[col].fillna(0)
    for side, cap in (("ab", "capacity_ab"), ("ba", "capacity_ba")):
        out[f"voc_{side}"] = (out[f"car_{side}"]
                              / (CAPACITY_FACTOR[day] * out[cap].replace(0, np.nan)))
    out["voc"] = out[["voc_ab", "voc_ba"]].max(axis=1).fillna(0)
    out["daily"] = out[CAR_AB] + out[CAR_BA]
    out["veh_h"] = (out[CAR_AB] * out["Congested_Time_AB"]
                    + out[CAR_BA] * out["Congested_Time_BA"]) / 60
    out["free_h"] = (out[CAR_AB] * out["travel_time_ab"]
                     + out[CAR_BA] * out["travel_time_ba"]) / 60
    return out


def totals(day):
    """One column of the comparison table."""
    d = loaded[day]
    return {"vehicle trips": DAY_MATRIX[day].sum(),
            "veh-km (thousands)": (d["daily"] * d["distance"]).sum() / 1e6,
            "veh-hours": d["veh_h"].sum(),
            "delay %": 100 * (d["veh_h"].sum() / d["free_h"].sum() - 1),
            "km over capacity": d.loc[d["voc"] >= 1, "distance"].sum() / 1000,
            "worst V/C": d["voc"].max()}


loaded = {d: streets(d) for d in DAYS}
days_table = pd.DataFrame({d: totals(d) for d in DAYS})
print(days_table.to_string(float_format=lambda v: f"{v:,.2f}"))

## C4. The days on a map

One map per day, coloured by daily V/C. The hover gives each street's daily flow.

In [ ]:
import shapely                                                        # noqa: E402

# Bands, colours and labels exactly as the road notebook draws them.
VOC_BANDS = [(0.00, 0.50, "#2c7bb6", "under 50% - free flowing"),
             (0.50, 0.85, "#8cc4de", "50-85% - busy"),
             (0.85, 1.00, "#fdae61", "85-100% - at capacity"),
             (1.00, 1.50, "#d7191c", "100-150% - over capacity"),
             (1.50, 1e9, "#7d0018", "over 150% - far beyond capacity")]
LINE_WEIGHT = {"motorway": 3.6, "trunk": 3.2, "primary": 2.6, "secondary": 2.0, "tertiary": 1.4}
MAP_ROAD_TYPES = ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link",
                  "secondary", "secondary_link", "tertiary", "tertiary_link"]
TIP_CSS = ("font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;"
           "background:white;padding:9px 12px;border:1px solid #999;border-radius:6px;")
BOX = ("position:fixed;bottom:24px;left:24px;z-index:9999;background:white;"
       "padding:9px 12px;border:1px solid #999;border-radius:6px;"
       "font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;")
PANEL = ("position:fixed;top:20px;right:20px;z-index:9999;background:white;"
         "padding:9px 12px;border:1px solid #999;border-radius:6px;"
         "font-family:sans-serif;font-size:13px;line-height:1.5;max-width:265px;")
# Esri's Light Gray Canvas: a pale base with its place labels as a separate layer. It needs no API
# key, only the attribution shown in the corner.
ESRI_CANVAS = ("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/{}/MapServer/tile/"
               "{{z}}/{{y}}/{{x}}")
ESRI_ATTRIBUTION = "Tiles &copy; Esri &mdash; Esri, HERE, Garmin, &copy; OpenStreetMap contributors"


def swatches():
    """The legend rows, drawn as the road notebook draws them."""
    return "".join(
        f'<div><span style="display:inline-block;width:16px;height:4px;background:{c};'
        f'margin:0 6px 3px 0;vertical-align:middle;"></span>{label}</div>'
        for _, _, c, label in VOC_BANDS)


def add_basemap(m):
    """Pale grey base and its labels, kept out of the layer control."""
    for layer, name in (("World_Light_Gray_Base", "Light grey"),
                        ("World_Light_Gray_Reference", "Place names")):
        folium.TileLayer(tiles=ESRI_CANVAS.format(layer), attr=ESRI_ATTRIBUTION, name=name,
                         max_zoom=16, control=False).add_to(m)
    return m


def band_colour(voc):
    for lo, hi, colour, _ in VOC_BANDS:
        if lo <= voc < hi:
            return colour
    return VOC_BANDS[-1][2]


with model.db_connection_spatial as _conn:
    _geo = pd.read_sql("SELECT link_id, AsBinary(geometry) wkb FROM links", _conn)
_geo["geometry"] = [shapely.wkb.loads(bytes(b)) for b in _geo["wkb"]]
_geo = gpd.GeoDataFrame(_geo.drop(columns="wkb"), geometry="geometry", crs="EPSG:4326")
_geo["geometry"] = _geo.to_crs(METRE_CRS).simplify(8).to_crs("EPSG:4326")
_geo["geometry"] = shapely.set_precision(_geo.geometry.values, 1e-5)
_centre = zones.to_crs("EPSG:4326").geometry.union_all().centroid


def map_day(day):
    """One day: each street coloured by its daily V/C, its daily figures on the hover."""
    d = loaded[day]
    d = d[(d["daily"] > 0) & d["link_type"].isin(MAP_ROAD_TYPES)].reset_index()
    drawn = gpd.GeoDataFrame(d.merge(_geo, on="link_id", how="inner"),
                             geometry="geometry", crs="EPSG:4326")
    base = drawn["link_type"].str.replace("_link", "", regex=False)
    drawn["colour"] = [band_colour(v) for v in drawn["voc"]]
    drawn["weight"] = (base.map(LINE_WEIGHT).fillna(1.4)
                       * np.where(drawn["link_type"].str.endswith("_link"), 0.6, 1.0))
    drawn["street"] = drawn["name"].fillna("unnamed").replace("", "unnamed")
    drawn["class"] = drawn["link_type"]
    drawn["lanes"] = drawn["lanes_ab"].fillna(0).astype(int)
    drawn["capacity veh/day"] = (CAPACITY_FACTOR[day] * drawn["capacity_ab"].fillna(0)).round(0)
    drawn["A to B veh/day"] = drawn[CAR_AB].round(0)
    drawn["B to A veh/day"] = drawn[CAR_BA].round(0)
    drawn["V/C"] = drawn["voc"].round(2)
    drawn["metres"] = drawn["distance"].round(0)
    fields = ["street", "class", "lanes", "capacity veh/day", "metres",
              "A to B veh/day", "B to A veh/day", "V/C", "link_id"]
    aliases = ["", "class", "lanes", "capacity veh/day", "metres",
               "A&rarr;B veh/day", "B&rarr;A veh/day", "V/C", "link"]

    m = folium.Map(location=[_centre.y, _centre.x], zoom_start=11, tiles=None,
                   prefer_canvas=True)
    add_basemap(m)
    folium.GeoJson(
        drawn[fields + ["colour", "weight", "geometry"]],
        name=day,
        style_function=lambda feature: {"color": feature["properties"]["colour"],
                                        "weight": feature["properties"]["weight"],
                                        "opacity": 0.85},
        highlight_function=lambda _feature: {"weight": 7, "opacity": 1},
        tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases, sticky=False,
                                      style=TIP_CSS),
    ).add_to(m)
    m.get_root().html.add_child(folium.Element(
        f'<div style="{BOX}"><b>Volume over capacity, daily</b>{swatches()}</div>'))
    col = days_table[day]
    over_km = drawn[drawn["voc"] >= 1.0]["distance"].sum() / 1000
    m.get_root().html.add_child(folium.Element(
        f'<div style="{PANEL}"><b>{day}, one assignment</b>'
        f'<div>{col["vehicle trips"]:,.0f} vehicle trips over {len(zones):,} zones</div>'
        f'<div>{col["veh-hours"]:,.0f} vehicle-hours, {col["delay %"]:.0f}% more '
        f'than the same trips on empty streets</div>'
        f'<div>{over_km:,.0f} of {drawn["distance"].sum() / 1000:,.0f} km drawn are over '
        f'capacity</div></div>'))
    filename = day.replace(" ", "_") + ".html"
    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"  {day:<13}{len(drawn):>7,} links   {filename:<18}"
          f"{os.path.getsize(path) / 1e6:>5.1f} MB")
    return m


for _day in DAYS:
    map_day(_day)

## C5. The matrices written out

Each day's O-D matrix, in two formats:

| file | what it holds |
|---|---|
| `weekday_matrix.omx`, `weekend_day_matrix.omx` | the matrix, square, in the format AequilibraE reads back |
| `weekday_matrix.csv.gz`, `weekend_day_matrix.csv.gz` | the same matrix, one row per zone pair that carries trips |

In [ ]:
MIN_TRIPS = 0.01          # a pair below this is rounding noise, not demand


def export_matrix(day):
    """Write a day's matrix as OMX, read it back, and write the pairs that carry trips."""
    stem = day.replace(" ", "_") + "_matrix"
    matrix = DAY_MATRIX[day]

    omx_path = os.path.join(EXPORT_DIR, stem + ".omx")
    out = AequilibraeMatrix()
    out.create_empty(zones=len(ZONE_IDS), matrix_names=["car"], index_names=["taz"],
                     memory_only=True)
    out.index[:] = ZONE_IDS
    out.matrices[:, :, 0] = matrix
    out.computational_view(["car"])
    out.export(omx_path)
    out.close()

    # Read it back, so a matrix that does not reopen is caught here.
    check = AequilibraeMatrix()
    check.create_from_omx(omx_path, os.path.join(tempfile.gettempdir(), f"_{stem}_check.aem"),
                          memory_only=True)
    if not np.array_equal(check.index[:], ZONE_IDS):
        raise ValueError(f"{stem}.omx does not carry the zone numbers")
    worst = float(np.abs(np.array(check.matrix["car"], dtype=float) - matrix).max())
    check.close()
    print(f"  {stem + '.omx':<26}{os.path.getsize(omx_path) / 1e6:>6.2f} MB   reopened, "
          f"largest difference {worst:.2e}")

    origin, dest = np.meshgrid(ZONE_IDS, ZONE_IDS, indexing="ij")
    carries = matrix > MIN_TRIPS
    pairs = pd.DataFrame({"origin": origin[carries], "destination": dest[carries],
                          "trips": matrix[carries].round(3)})
    csv_path = os.path.join(EXPORT_DIR, stem + ".csv.gz")
    pairs.to_csv(csv_path, index=False, compression="gzip")
    print(f"  {stem + '.csv.gz':<26}{os.path.getsize(csv_path) / 1e6:>6.2f} MB   "
          f"{len(pairs):,} pairs holding {100 * pairs['trips'].sum() / matrix.sum():.2f}% "
          f"of the trips")


for _day in DAYS:
    export_matrix(_day)

## C6. What is assumed

Network, zones, the morning peak matrix, productions and `beta` come from the road model
unchanged. This notebook adds:

| | |
|---|---|
| `WEEKDAY_SIZE` | 10.16 morning-peak hours of traffic in a weekday |
| `SATURDAY, SUNDAY` | 0.80 and 0.65 of a weekday's traffic |
| `WEEKEND_RATES` | relative pull per m² of each floor-area class on a weekend |
| `BUSIEST_HOUR_SHARE` | 1 / 10.16 = 9.8% on a weekday, 7.7% on a weekend day; sets `F` |
| `BPR_ALPHA, BPR_BETA` | 0.15 and 4.0, BPR's published values for one hour |

### What this does not capture

- **No direction.** A day's matrix is symmetric, so neither peak's one-way loading appears.
- **Congestion is the same all day.** Every hour is the busiest hour. Delay depends on `F` more
  than on anything else.
- **No hours.** No busiest hour per street and no hours over capacity.
- **The day sizes and busiest hours are not measured.** One count station with an hourly profile,
  on a weekday and at a weekend, would set them.
- **The weekend rates are not measured.** Weekend traffic counts at shopping centres and in the
  historic centre would set them.
- **Shops and offices are one class.** `mixed/commercial` cannot tell a shopping centre from an
  office block, so offices still pull on a weekend.
- **The same trip lengths.** `beta` is the weekday's, so weekend trips are only as long as their
  new destinations make them.
- **BPR is set for one hour.** A daily model would fit its own `alpha` and `beta` to observed
  daily speeds.